In [1]:
# =========================
# 1️⃣ 라이브러리
# =========================
import pandas as pd
import joblib
import pickle
from pathlib import Path

# =========================
# 2️⃣ 경로 설정
# =========================
BASE_DIR = Path().resolve().parent

DATA_PATH = BASE_DIR / "data" / "cc_fraud_train_processed.csv"
MODEL_PATH = BASE_DIR / "model" / "xgboost_tuned.pkl"
OUTPUT_PATH = BASE_DIR / "data" / "ml_scored_data.csv"

print("📂 BASE_DIR:", BASE_DIR)
print("📂 MODEL_PATH:", MODEL_PATH)
print("📂 MODEL 존재 여부:", MODEL_PATH.exists())

# =========================
# 3️⃣ 데이터 로딩
# =========================
print("📥 데이터 로딩")
df = pd.read_csv(DATA_PATH)
print("데이터 shape:", df.shape)

# =========================
# 4️⃣ 모델 로드
# =========================
print("🤖 모델 로딩")

try:
    model = joblib.load(MODEL_PATH)
except:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)

print("✅ 모델 로드 완료:", type(model))

# =========================
# 5️⃣ 컬럼 추출 (모델에서 직접)
# =========================
cols = None

# 방법 A: XGBoost Booster feature_names
try:
    cols = model.get_booster().feature_names
    print("✅ [방법A] get_booster().feature_names 추출:", len(cols))
except Exception as e:
    print("❌ 방법A 실패:", e)

# 방법 B: sklearn API feature_names_in_
if cols is None:
    try:
        cols = list(model.feature_names_in_)
        print("✅ [방법B] feature_names_in_ 추출:", len(cols))
    except Exception as e:
        print("❌ 방법B 실패:", e)

# 방법 C: 데이터 컬럼에서 타겟/불필요 컬럼 제외
if cols is None:
    print("⚠️ 모델에서 컬럼 추출 실패 → 데이터 컬럼 자동 사용")
    exclude_cols = ["Class", "ml_score", "ml_score_scaled"]  # 타겟/결과 컬럼
    cols = [c for c in df.columns if c not in exclude_cols]
    print("✅ [방법C] 데이터 컬럼 사용:", len(cols))

print("컬럼 예시 (앞 5개):", cols[:5])

# =========================
# 6️⃣ feature 맞추기
# =========================
# 데이터에 없는 컬럼 확인
missing = [c for c in cols if c not in df.columns]
if missing:
    print(f"⚠️ 데이터에 없는 컬럼 {len(missing)}개: {missing[:5]}")

X = df[[c for c in cols if c in df.columns]]
print("최종 feature shape:", X.shape)

# =========================
# 7️⃣ ML Score 생성
# =========================
print("⚙️ ml_score 생성")

df["ml_score"] = model.predict_proba(X)[:, 1]
df["ml_score_scaled"] = (df["ml_score"] * 100).round(3)

# =========================
# 8️⃣ 결과 확인
# =========================
print(df[["ml_score", "ml_score_scaled"]].describe())

# =========================
# 9️⃣ 저장
# =========================
df.to_csv(OUTPUT_PATH, index=False)
print("✅ 저장 완료:", OUTPUT_PATH)

📂 BASE_DIR: C:\Users\kim\Desktop\2ndprj\fold\Teamproject (1)
📂 MODEL_PATH: C:\Users\kim\Desktop\2ndprj\fold\Teamproject (1)\model\xgboost_tuned.pkl
📂 MODEL 존재 여부: True
📥 데이터 로딩
데이터 shape: (1296675, 22)
🤖 모델 로딩


c:\Users\kim\anaconda3\envs\VM_01\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\kim\anaconda3\envs\VM_01\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\kim\anaconda3\envs\VM_01\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.1 when using version 1.7.2. This mi

✅ 모델 로드 완료: <class 'sklearn.pipeline.Pipeline'>
❌ 방법A 실패: 'Pipeline' object has no attribute 'get_booster'
✅ [방법B] feature_names_in_ 추출: 21
컬럼 예시 (앞 5개): ['merchant', 'category', 'amt', 'gender', 'state']
최종 feature shape: (1296675, 21)
⚙️ ml_score 생성
           ml_score  ml_score_scaled
count  1.296675e+06     1.296675e+06
mean   6.318498e-03     6.317971e-01
std    7.620040e-02     7.620046e+00
min    1.375839e-10     0.000000e+00
25%    8.457677e-07     0.000000e+00
50%    3.846769e-06     0.000000e+00
75%    1.880495e-05     2.000000e-03
max    1.000000e+00     1.000000e+02
✅ 저장 완료: C:\Users\kim\Desktop\2ndprj\fold\Teamproject (1)\data\ml_scored_data.csv
